# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a workflow for loading and exploring the published FAIR\u00b2 dataset, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL (Croissant schema)
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load and display the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {', '.join(metadata.keywords)}\n")

## 2. Data Overview
Review available record sets, their IDs, associated fields, and column structure. We reference all entities by their `@id` from the underlying Croissant schema.

In [ ]:
# List all RecordSets and their fields (referenced by their @id)
from pprint import pprint

record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    # Sometimes 'recordSet' is top-level, sometimes under 'dataset', sometimes pluralized
    print("No record sets declared in the root metadata. Trying group under 'dataset', 'hasPart', or other top-level structures...")

# Print all available RecordSets with their @id and name
if record_sets:
    print(f"Found {len(record_sets)} RecordSet(s) in the dataset:\n")
    for rs in record_sets:
        # Each record set is a dict with @id
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs)
        print(f"- @id: {rs_id}")
else:
    print("Record sets information not found or empty in Croissant schema. Trying to auto-discover using the records() generator.")
    # Use mlcroissant's Dataset API to enumerate possible record sets
    possible_record_sets = [rs for rs in dataset._manifest._recordsets]
    print(f"Discovered {len(possible_record_sets)} record set(s):")
    for rs in possible_record_sets:
        print(f"- @id: {rs['@id']}, name: {rs.get('name', '')}")
    # For this dataset, we'll use the discovered RecordSet IDs
    record_sets = [{'@id': rs['@id'], 'name': rs.get('name', '')} for rs in possible_record_sets]
    # Show fields for each
    for rs in possible_record_sets:
        print(f"\nFields in RecordSet {rs['@id']} ({rs.get('name','')}):")
        for f in rs.get('field', []):
            if isinstance(f, dict):
                print(f"  - @id: {f['@id']}, name: {f.get('name', '')}, dataType: {f.get('dataType', '')}")
            else:
                print(f"  - field @id: {f}")

## 3. Data Extraction
Load data from the record set(s) into pandas DataFrames for analysis. Use the record set and field `@id`s printed above.

In [ ]:
# Define record set IDs to extract (replace with actual IDs printed above)
# In this dataset, the main data recordset holds protein table; let's try to extract all discovered record sets
main_recordset_id = record_sets[0]['@id'] if record_sets else 'cr:RecordSet/proteins'
print(f"Selected main record set @id: {main_recordset_id}")

dataframes = {}
for rs in record_sets:
    rsid = rs['@id']
    print(f"Loading records for RecordSet @id: {rsid}")
    # Each record is a dictionary with field @id as the key
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"\u25cf DataFrame columns for RecordSet {rsid}:\n", df.columns.tolist())
    print(df.head(2))

# Focus on the main record set and examine columns
print(f"\nMain RecordSet ({main_recordset_id}) columns:")
print(dataframes[main_recordset_id].columns.tolist())
dataframes[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter, normalize, or group by certain fields. All fields are referenced by their `@id`. For demonstration, we select a numeric field (e.g., peptide count, abundance, etc) and a group field for summarization.

In [ ]:
# Select a numeric field's @id (e.g., cr:Field/peptide_count) and a group field
# Get a list of field @ids from the dataframe (these match the actual @id values)
numeric_field = None
group_field = None
for col in dataframes[main_recordset_id].columns:
    cname = str(col).lower()
    if 'count' in cname or 'abundance' in cname or 'coverage' in cname or 'mw' in cname:
        numeric_field = col
        break
for col in dataframes[main_recordset_id].columns:
    if 'description' in col or 'group' in col or 'type' in col or 'category' in col:
        group_field = col
        break

print(f"Using numeric field @id: {numeric_field}")
if group_field is not None:
    print(f"Using group field @id: {group_field}")

if numeric_field is not None:
    try:
        # Attempt conversion to numeric (in-place)
        dataframes[main_recordset_id][numeric_field] = pd.to_numeric(dataframes[main_recordset_id][numeric_field], errors='coerce')
        threshold = dataframes[main_recordset_id][numeric_field].mean() # demo: use mean as threshold
        filtered_df = dataframes[main_recordset_id][dataframes[main_recordset_id][numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field is not None and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nMean {numeric_field} grouped by {group_field}:")
            print(grouped_df.head())
    except Exception as e:
        print(f"Error during processing: {e}")
else:
    print("No numeric field found via @id heuristics.")

## 5. Visualization
Visualize the distribution of a numeric field or its relationship to a group field using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(dataframes[main_recordset_id][numeric_field].dropna(), bins=40, color='skyblue')
    plt.title(f'Distribution for field {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field and group_field in dataframes[main_recordset_id].columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_recordset_id])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and analyzing the FAIR\u00b2 mass spectrometry dataset using the Croissant schema and `mlcroissant`. You explored record structure, loaded data by @id, processed and visualized numeric protein fields, and experimented with grouping or filtering—all referenced by stable `@id` as encouraged for FAIR workflows.

For deep exploration, refer to the dataset schema and scientific publication, and extend this notebook with domain-specific visualizations or downstream analysis.